In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Create Widgets ---
dbutils.widgets.removeAll()
dbutils.widgets.text("scale", "1000", "Scale Factor (GB)")
dbutils.widgets.text("sf_account_locator", "", "Snowflake Account Locator")
dbutils.widgets.text("sf_region", "", "Snowflake Region")
dbutils.widgets.text("sf_warehouse", "", "Snowflake Warehouse")
dbutils.widgets.text("sf_database", "", "Snowflake Database")
dbutils.widgets.text("sf_schema", "PUBLIC", "Snowflake Schema")
dbutils.widgets.text("sf_role", "PUBLIC", "Snowflake Role")
dbutils.widgets.text("sf_user", "", "Snowflake User")
dbutils.widgets.text("sf_hourly_rate", "4.00", "Hourly Rate")
dbutils.widgets.text("mlflow_experiment_name", "/TPC-DS Benchmark", "MLflow Experiment")
dbutils.widgets.text("run_ingestion", "true", "Run Ingestion Test")
dbutils.widgets.text("run_power_test", "true", "Run Power Test")
dbutils.widgets.text("run_concurrency", "true", "Run Concurrency Test")

In [ ]:
import os

params = dbutils.widgets.getAll()
current_path = os.getcwd()

# The RSA key was written by the Snowflake JMX Generation notebook (4.1)
auth_key_path = os.path.join(current_path, "test_conf", "rsa_key.p8")

# Fetch live warehouse configuration from Snowflake
warehouse_config = get_snowflake_warehouse_config(
    params["sf_account_locator"],
    params["sf_region"],
    params["sf_warehouse"],
    params["sf_user"],
    auth_key_path,
    params["sf_role"],
    params["sf_database"],
    params["sf_schema"],
)
print(f"Warehouse: {warehouse_config}")

# MLflow parameters shared by all runs in this notebook
mlflow_params = {
    "scale": params["scale"],
    "warehouse_name": warehouse_config["warehouse_name"],
    "warehouse_size": warehouse_config["warehouse_size"],
    "warehouse_type": warehouse_config["warehouse_type"],
    "hourly_rate": params["sf_hourly_rate"],
}

In [ ]:
%sh
set -e
WS_DIR="$PWD"
TPCDS_DIR="/tmp/tpcds"

# Prepare a local working directory (Workspace FUSE doesn't reliably persist files written by JMeter)
# Reuse the same /tmp/tpcds dir - setup cell from notebook 5 may have already created it
mkdir -p "$TPCDS_DIR/results" "$TPCDS_DIR/logs" "$TPCDS_DIR/test_conf"

cp "$WS_DIR"/test_conf/*.jmx "$TPCDS_DIR/test_conf/"
cp "$WS_DIR"/test_conf/*.tsv "$TPCDS_DIR/test_conf/"
cp "$WS_DIR"/test_conf/*.p8 "$TPCDS_DIR/test_conf/" 2>/dev/null || true
cp -r "$WS_DIR"/tpcds_snowflake_concurrency_streams "$TPCDS_DIR/" 2>/dev/null || true

# Rewrite workspace paths in JMX files to use the local directory
sed -i "s|${WS_DIR}|${TPCDS_DIR}|g" "$TPCDS_DIR"/test_conf/*.jmx

echo "Prepared local working directory at $TPCDS_DIR"

In [ ]:
if params.get("run_ingestion", "true").lower() == "true":
    run_jmeter_with_tracking(
        platform="snowflake",
        test_type="ingestion",
        jmx_file="/tmp/tpcds/test_conf/snowflake_ingestion.jmx",
        tsv_files=[
            "/tmp/tpcds/test_conf/tpcds-queries-ingestion-snowflake.tsv",
            "/tmp/tpcds/test_conf/tpcds-queries-optimization-snowflake.tsv",
        ],
        jdbc_driver="/tpcds/sf_driver.jar",
        result_csvs=[
            "/tmp/tpcds/results/snowflake-ingestion-report.csv",
            "/tmp/tpcds/results/snowflake-optimize-report.csv",
        ],
        log_file="/tmp/tpcds/logs/jmeter-sf-ingestion.log",
        run_params=mlflow_params,
        experiment_name=params["mlflow_experiment_name"],
        workspace_results_dir=f"{current_path}/results",
        workspace_logs_dir=f"{current_path}/logs",
    )
else:
    print("Skipping ingestion test (run_ingestion=false)")

In [ ]:
if params.get("run_power_test", "true").lower() == "true":
    run_jmeter_with_tracking(
        platform="snowflake",
        test_type="power_test",
        jmx_file="/tmp/tpcds/test_conf/snowflake_power_test.jmx",
        tsv_files=[
            "/tmp/tpcds/test_conf/tpcds-queries-perf-snowflake.tsv",
        ],
        jdbc_driver="/tpcds/sf_driver.jar",
        result_csvs=[
            "/tmp/tpcds/results/snowflake-perf-test-report.csv",
        ],
        log_file="/tmp/tpcds/logs/jmeter-sf-power-test.log",
        run_params=mlflow_params,
        experiment_name=params["mlflow_experiment_name"],
        workspace_results_dir=f"{current_path}/results",
        workspace_logs_dir=f"{current_path}/logs",
    )
else:
    print("Skipping power test (run_power_test=false)")

In [ ]:
if params.get("run_concurrency", "true").lower() == "true":
    import glob
    
    concurrency_streams = sorted(
        glob.glob("/tmp/tpcds/tpcds_snowflake_concurrency_streams/stream_*.tsv")
    )
    
    run_jmeter_with_tracking(
        platform="snowflake",
        test_type="concurrency",
        jmx_file="/tmp/tpcds/test_conf/snowflake_concurrency.jmx",
        tsv_files=concurrency_streams,
        jdbc_driver="/tpcds/sf_driver.jar",
        result_csvs=[
            "/tmp/tpcds/results/snowflake-concurrency-report.csv",
        ],
        log_file="/tmp/tpcds/logs/jmeter-sf-concurrency.log",
        run_params=mlflow_params,
        experiment_name=params["mlflow_experiment_name"],
        workspace_results_dir=f"{current_path}/results",
        workspace_logs_dir=f"{current_path}/logs",
    )
else:
    print("Skipping concurrency test (run_concurrency=false)")